In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
superpotato9_dalle_recognition_dataset_path = kagglehub.dataset_download('superpotato9/dalle-recognition-dataset')

print('Data source import complete.')



Using Colab cache for faster access to the 'dalle-recognition-dataset' dataset.
Data source import complete.


In [2]:
!ls /kaggle/input/

dalle-recognition-dataset


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import ReduceLROnPlateau
import timm  # for EfficientNet
import os

# from tensorflow.keras.models import Model, load_model
# from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from tensorflow.keras.applications import (
#     MobileNet, MobileNetV2, DenseNet121, InceptionV3, ResNet50, EfficientNetB0
# )
# from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.image as mpimg

import joblib

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [4]:
data_dir = '/kaggle/input'
fake_dir = os.path.join(data_dir, '/kaggle/input/dalle-recognition-dataset/fakeV2/fake-v2')
real_dir = os.path.join(data_dir, '/kaggle/input/dalle-recognition-dataset/real')


# Create DataFrame with image paths and labels
def create_dataframe(fake_dir, real_dir):
    fake_images = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir)
                  if f.endswith(('.jpg', '.jpeg', '.png'))]
    real_images = ([os.path.join(real_dir, f) for f in os.listdir(real_dir)
                   if f.endswith(('.jpg', '.jpeg', '.png'))])

    fake_labels = ['AiArt'] * len(fake_images)
    real_labels = ['RealArt'] * len(real_images)

    data = pd.DataFrame({
        'filename': fake_images + real_images,
        'class': fake_labels + real_labels
    })
    return data

# Create and balance dataset
data = create_dataframe(fake_dir, real_dir)
print(data['class'].value_counts())
data['label'] = np.where(data['class']=="RealArt",1,0)

class
AiArt      17855
RealArt     3780
Name: count, dtype: int64


In [22]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class ArtDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, 'filename']
        label = self.dataframe.loc[idx, 'label']

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long), img_path

from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])



In [24]:
from sklearn.model_selection import train_test_split

# 20% test
train_df, test_df = train_test_split(
    data,
    test_size=0.2,
    stratify=data['label'],
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.25,   # 0.25 * 0.8 = 0.20
    stratify=train_df['label'],
    random_state=42
)



In [25]:
from torch.utils.data import DataLoader

train_dataset = ArtDataset(train_df, transform=train_transforms)
val_dataset = ArtDataset(val_df, transform=val_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_dataset = ArtDataset(test_df, transform=val_transforms)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [11]:
import timm
import torch
from PIL import Image # Import Image to modify its global setting

# Set MAX_IMAGE_PIXELS to None to disable the decompression bomb check
Image.MAX_IMAGE_PIXELS = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 2  # AiArt / RealArt

model = timm.create_model(
    "vit_base_patch32_224",
    pretrained=True,
    num_classes=num_classes
)

model.to(device)

import torch.optim as optim
from tqdm.auto import tqdm

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-5,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

def train_vit(model, train_loader, val_loader, epochs):

    best_val_acc = 0

    for epoch in range(epochs):
        # -------- TRAIN --------
        model.train()
        train_loss = 0
        correct = 0
        total = 0

        for images, labels, _ in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total

        # -------- VALIDATION --------
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels , _ in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"):
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_acc = correct / total

        scheduler.step()

        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Train Acc: {train_acc:.4f}")
        print(f"Val Acc: {val_acc:.4f}")
        print("-" * 30)

        # Save best
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_vit.pth")
            print("Saved Best Model ✅")


train_vit(model, train_loader, val_loader, epochs=1)

Epoch 1 Training:   0%|          | 0/406 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [13]:
!pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.4 MB/s eta 0:00:00


In [14]:
import open_clip

model_clip, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32',
    pretrained='openai'
)

tokenizer = open_clip.get_tokenizer('ViT-B-32')

model_clip.to(device)


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [20]:
import torch.nn.functional as F
model_clip.to(device)
model_clip.eval()

# Extract text encoder
text_encoder = model_clip.encode_text

# Get tokenizer
tokenizer = open_clip.get_tokenizer("ViT-B-32")


class_prompts = [
    "AI generated artwork",
    "Real artwork"
]

text_tokens = tokenizer(class_prompts).to(device)

with torch.no_grad():
    text_features = text_encoder(text_tokens)
    text_features = F.normalize(text_features, dim=-1)

print(text_features.shape)  # [num_classes, 512]

image_encoder = model_clip.visual
# image_encoder.load_state_dict(torch.load("/content/best_vit.pth"), strict=False)
image_encoder.to(device)
# image_encoder =
image_encoder.eval()

def test_custom_clip(image_encoder, text_encoder, test_loader, text_features):
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels , path in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            img_features = image_encoder(images)
            img_features = F.normalize(img_features, dim=-1)

            text_features = F.normalize(text_features, dim=-1)

            logits = img_features @ text_features.T
            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print(f"Custom CLIP Accuracy: {correct/total:.4f}")



torch.Size([2, 512])


In [21]:
test_custom_clip(image_encoder, text_encoder, test_loader, text_features)

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Custom CLIP Accuracy: 0.5845


In [ ]:
import pandas as pd
import os

def test_custom_clip_collect_failures(
    image_encoder,
    text_features,
    test_loader,
    device,
    save_images=False,
    save_dir="failure_cases"
):
    image_encoder.eval()

    correct = 0
    total = 0

    failures = []

    if save_images:
        os.makedirs(save_dir, exist_ok=True)

    with torch.no_grad():
        for images, labels, paths in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            img_features = image_encoder(images)
            img_features = F.normalize(img_features, dim=-1)

            normalized_text = F.normalize(text_features, dim=-1)

            logits = img_features @ normalized_text.T
            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # Identify misclassified samples
            mismatches = preds != labels

            for i in range(len(images)):
                if mismatches[i]:

                    failure_info = {
                        "filepath": paths[i],
                        "true_label": labels[i].item(),
                        "predicted_label": preds[i].item(),
                        "confidence": torch.softmax(logits[i], dim=0)[preds[i]].item()
                    }

                    failures.append(failure_info)

                    if save_images:
                        filename = os.path.basename(paths[i])
                        save_path = os.path.join(save_dir, filename)

                        # Save original image (reload for clean save)
                        original = Image.open(paths[i]).convert("RGB")
                        original.save(save_path)

    accuracy = correct / total
    print(f"Custom CLIP Accuracy: {accuracy:.4f}")
    print(f"Total Failures Collected: {len(failures)}")

    failures_df = pd.DataFrame(failures)
    failures_df.to_csv("failure_cases.csv", index=False)

    return failures_df


failures_df = test_custom_clip_collect_failures(image_encoder,
    text_features,
    test_loader,
    device,
    save_images=True,
    save_dir="failure_cases")

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [ ]:
from sklearn.utils import resample

# Split by class
class_counts = data['class'].value_counts()
max_class = class_counts.idxmax()
min_class = class_counts.idxmin()
max_count = class_counts.max()
min_count = class_counts.min()

# Extract classes
df_majority = data[data['class'] == max_class]
df_minority = data[data['class'] == min_class]

# Undersample majority class (down to average size or desired count)
df_majority_downsampled = resample(
    df_majority,
    replace=False,
    n_samples=6000,
    random_state=42
)

# Oversample minority class (up to same size as majority sample)
df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=6000,
    random_state=42
)

# Combine them
balanced_data = pd.concat([df_majority_downsampled, df_minority_upsampled]).sample(frac=1, random_state=42).reset_index(drop=True)
print(balanced_data['class'].value_counts())

In [ ]:
print(balanced_data.columns)


In [ ]:
print(balanced_data.head())


In [ ]:
classes = balanced_data['class'].unique()

# Loop through each class
for cls in classes:
    # Get 5 sample images for this class
    sample_rows = balanced_data[balanced_data['class'] == cls].head(5)

    # Create a new figure for each class
    plt.figure(figsize=(15, 3))  # Width=5*3 inches, Height=3 inches
    plt.suptitle(f'Class: {cls}', fontsize=16)

    for i, (_, row) in enumerate(sample_rows.iterrows()):
        img = mpimg.imread(row['filename'])

        plt.subplot(1, 5, i + 1)
        plt.imshow(img)
        plt.title(f'Sample {i + 1}')
        plt.axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.9])  # Leave space for title
    plt.show()

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# ImageDataGenerators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3],  # Increased range
    channel_shift_range=70.0,     # Increased range
    fill_mode='reflect',
    featurewise_center=True,      # Add normalization
    featurewise_std_normalization=True
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
# Split into train, validation, test sets
train_val, test_data = train_test_split(balanced_data, test_size=0.10, random_state=42, stratify=balanced_data['class'])
train_data, val_data = train_test_split(train_val, test_size=0.11, random_state=42, stratify=train_val['class'])


print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")
print(f"Test set size: {len(test_data)}")



In [ ]:
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_data,
    x_col="filename",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_data,
    x_col="filename",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_data,
    x_col="filename",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

# Optional: View class mappings
class_names = train_generator.class_indices
print(f"Class names: {class_names}")